# Submissão B — PyTorch GRU

Classificação do `subm1.csv` com o modelo GRU treinado em PyTorch.

In [13]:
import re, pickle
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}')


Device: cuda


In [14]:
class GRUClassifier(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim, num_classes,
                 num_layers=1, dropout=0.3):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.rnn = nn.GRU(
            input_size  = embed_dim,
            hidden_size = hidden_dim,
            num_layers  = num_layers,
            batch_first = True,
            dropout     = dropout if num_layers > 1 else 0.0
        )
        self.dropout = nn.Dropout(dropout)
        self.fc      = nn.Linear(hidden_dim, num_classes)

    def forward(self, x):
        emb        = self.embedding(x)
        _, h       = self.rnn(emb)
        return self.fc(self.dropout(h[-1]))

print('GRUClassifier definido.')

GRUClassifier definido.


In [15]:
class SequenceDataset(Dataset):
    def __init__(self, texts, labels, word_index, max_len):
        self.labels = torch.tensor(labels, dtype=torch.long)
        data = np.zeros((len(texts), max_len), dtype=np.int64)
        for i, text in enumerate(texts):
            toks = text.split()[:max_len]
            for j, tok in enumerate(toks):
                data[i, j] = word_index.get(tok, 1)
        self.X = torch.tensor(data, dtype=torch.long)

    def __len__(self): return len(self.X)
    def __getitem__(self, idx): return self.X[idx], self.labels[idx]

print('SequenceDataset definido.')


SequenceDataset definido.


## Classificação e geração do ficheiro de submissão

In [16]:
# -- 1. Carregar artefactos ----------------------------------------------------
with open('torch_word_index.pkl', 'rb') as f:
    vocab_rnn = pickle.load(f)

with open('torch_idx2label.pkl', 'rb') as f:
    IDX_TO_CLASS = pickle.load(f)

VOCAB_SIZE_RNN = len(vocab_rnn)
N_CLASSES      = len(IDX_TO_CLASS)
MAX_LEN        = 180
BATCH_SIZE     = 256
print(f'Vocab: {VOCAB_SIZE_RNN}  Classes: {IDX_TO_CLASS}')

# -- 2. Carregar e preparar subm1.csv ------------------------------------------
def clean_text(text):
    text = str(text).lower()
    text = re.sub(r'<.*?>', ' ', text)
    text = re.sub(r'[^\w\s]', ' ', text, flags=re.UNICODE)
    return re.sub(r'\s+', ' ', text).strip()

df = pd.read_csv('subm1.csv', sep=';', encoding='utf-8-sig')
df.columns = df.columns.str.strip()
df['Text'] = df['Text'].astype(str).apply(clean_text)
print(f'Registos: {len(df)}')

# -- 3. Tokenizar --------------------------------------------------------------
dummy_labels = [0] * len(df)
seq_loader   = DataLoader(
    SequenceDataset(df['Text'].tolist(), dummy_labels, vocab_rnn, MAX_LEN),
    batch_size=BATCH_SIZE
)

# -- 4. Carregar GRU -----------------------------------------------------------
gru_model = GRUClassifier(
    vocab_size=VOCAB_SIZE_RNN, embed_dim=128, hidden_dim=256,
    num_classes=N_CLASSES, num_layers=1, dropout=0.3
)
gru_model.load_state_dict(torch.load('gru_model.pth', map_location=DEVICE))
gru_model.to(DEVICE)
gru_model.eval()
print('Modelo GRU carregado.')

# -- 5. Previsões --------------------------------------------------------------
all_preds = []
with torch.no_grad():
    for x, _ in seq_loader:
        all_preds.extend(gru_model(x.to(DEVICE)).argmax(1).cpu().tolist())

labels_pred = [IDX_TO_CLASS[i] for i in all_preds]

# -- 6. Guardar CSV ------------------------------------------------------------
df_out = pd.DataFrame({'ID': df['ID'], 'Labels': labels_pred})
df_out.to_csv('subm1-g5-MECD-B.csv', index=False, sep=';')
print('\nFicheiro guardado: subm1-g5-MECD-B.csv')
print(df_out['Labels'].value_counts().to_string())
print(df_out.head(10).to_string(index=False))


Vocab: 20000  Classes: {0: 'Anthropic', 1: 'Google', 2: 'Human', 3: 'Meta', 4: 'Openai'}
Registos: 150
Modelo GRU carregado.

Ficheiro guardado: subm1-g5-MECD-B.csv
Labels
Openai       59
Google       36
Human        21
Meta         19
Anthropic    15
   ID    Labels
 D2-1    Google
 D2-2    Google
 D2-3    Google
 D2-4      Meta
 D2-5     Human
 D2-6    Openai
 D2-7 Anthropic
 D2-8    Google
 D2-9 Anthropic
D2-10     Human
